# Fine-tuning OWSM on a custom dataset with ESPnet3

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/espnet/notebook/blob/master/Courses/CMUSpeechTechnology26S/owsm_finetuning.ipynb) [![owsm_finetuning](https://github.com/espnet/notebook/actions/workflows/owsm_finetuning.yml/badge.svg)](https://github.com/espnet/notebook/actions/workflows/owsm_finetuning.yml)

Take a pretrained OWSM model, fine-tune it on one language of FLEURS with the
ESPnet3 trainer, and compare the transcripts before and after.

This is the demonstration part of an assignment from CMU 11492/11692/18495,
*Speech Technology for Conversational AI*, kept here without the graded
exercises so that it runs end to end.

Author: Masao Someki

Main references:
- [ESPnet repository](https://github.com/espnet/espnet)
- [ESPnet3 recipes](https://github.com/espnet/espnet/tree/master/egs3)
- [OWSM](https://www.wavlab.org/activities/2024/owsm/)

## 1) Prerequisites (3 minutes)

### Environment setup

- This is a full installation method to perform data preprocessing, training and inference.

- We prepare various ways of installation. Please read https://espnet.github.io/espnet/installation.html#step-2-installation-espnet for more details.

- We also have some other toolkits/packages needed for this assignment.

The training stack comes with the `[train]` extra: Lightning, Hydra and
`datasets`, which is what ESPnet3's trainer is built on.

The pin matters more here than in an inference notebook. Fine-tuning reads a
config that the trainer's API has to agree with, and a release is a fixed
agreement; `master` is not.

In [ ]:
%pip install -q "espnet[train]==202610.post1"

## 2) Imports

### Runtime imports


In [ ]:
import os
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from datasets import Audio as AudioFeature, load_dataset
from omegaconf import OmegaConf
from torch.utils.data import Dataset
from espnet2.bin.s2t_inference import Speech2Text


```
# This is formatted as code
```

## 3) Data Processing (3 minutes)

For this tutorial, we will use the [FLEURS](https://arxiv.org/abs/2205.12446) dataset from HuggingFace: https://huggingface.co/datasets/google/fleurs .

FLEURS is a 102-language multilingual speech dataset, supporting tasks such as Automatic Speech Recognition (ASR), Speech Translation (ST), and Language Identification (LID).

While the total size of FLEURS is relatively large at ~1000 hours of training data, each individual language only has 7-10 hours of audio.

For this tutorial, we will focus on monolingual ASR for one of the 102 languages.


### Data Downloading

We will first download the data for one language of FLEURS. FLEURS organizes the languages by its ISO2 language code and locale. For example, American English is `en_us`.

**We will use English for the first fine-tuning experiment.** You will have the opportunity to try a different language later on in the assignment.

If you want to download the data for another language, you can map the language name to the ISO2 code using Table 9 in the FLEURS paper: https://arxiv.org/pdf/2205.12446. Then, you can use that to identify the language+region combination using the HuggingFace data previewer: https://huggingface.co/datasets/google/fleurs .

(Please select y if you see the prompt of running custom code to download the data)


🔭 Exploration point:
*   trying different langauge splits of FLEURS


In [ ]:
# Fine-tuning is the one thing in these notebooks that a free CI runner
# cannot do at full size, so the weekly run shrinks it through the
# environment. Unset - which is what you have when you open this - every one
# of these is the real thing.
import os

LIMIT_TRAIN_BATCHES = int(os.environ.get("LIMIT_TRAIN_BATCHES", 300))
# the splits the trainer and the test section use; a slice such as
# "validation[:8]" is a smaller download as well as a shorter run
TRAIN_SPLIT = os.environ.get("FLEURS_TRAIN_SPLIT", "train")
VALID_SPLIT = os.environ.get("FLEURS_VALID_SPLIT", "validation")
TEST_SPLIT = os.environ.get("FLEURS_TEST_SPLIT", "test")

FLEURS_CONFIG = "en_us"

import io

import soundfile
from datasets import Audio as AudioFeature
from datasets import load_dataset
from IPython.display import Audio, display

# decode=False keeps the audio as bytes: decoding through datasets needs
# torchcodec, and soundfile reads the bytes directly. One split rather than
# all of them: en_us is 4.2 GB in full, and this is only to look at a row.
fleurs_train = load_dataset(
    "google/fleurs", FLEURS_CONFIG, split=TRAIN_SPLIT
).cast_column("audio", AudioFeature(decode=False))
print(TRAIN_SPLIT, len(fleurs_train), "utterances")

### Inspect the Data


In [ ]:
row = fleurs_train[0]
print(row["transcription"])
print(row["audio"]["path"])


In [ ]:
row = fleurs_train[0]
speech, fs = soundfile.read(io.BytesIO(row["audio"]["bytes"]), dtype="float32")
print(row["transcription"])
display(Audio(speech, rate=fs))


## 4) Workspace, pre-trained model, and tokenizer assets

In low-resource settings, training a model from scratch is unlikely to lead to good results. So instead, we will fine-tune a pre-trained foundation model.
We will use the base version of [OWSM 3.1](https://arxiv.org/pdf/2401.16658), an open-source speech foundation model trained on 180K hours of multilingual ASR and ST.

Here we also set the path `WORK_DIR`, in which dataset, checkpoints and logs will be saved.


### Downloading

Since it needs to support many language varieties, OWSM uses ISO3 for the language IDs. The ISO3 code for your language of choice can also be found in Table 9 in the FLEURS paper: https://arxiv.org/pdf/2205.12446


🔭 Exploration point:
*   Trying different pretrained models
*   Remember to align the language of `OWSM_LANG` and `FLEURS_CONFIG`


In [ ]:
MODEL_TAG = "espnet/owsm_v3.1_ebf_base"     # You can also change to OWSM v4: "espnet/owsm_v4_base_102M"
OWSM_LANG = "eng"  # ISO3 (e.g., eng, jpn)

In [ ]:
import torch
WORK_DIR = Path(os.environ.get("WORK_DIR", "./work/owsm_v31_fleurs")).resolve()
WORK_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

s2t = Speech2Text.from_pretrained(
    model_tag=MODEL_TAG,
    lang_sym=f"<{OWSM_LANG}>",
    device="cuda" if torch.cuda.is_available() else "cpu",
)
torch.save(s2t.s2t_model.state_dict(), 'original.pth')

BPEMODEL = s2t.tokenizer.model
TOKEN_LIST = WORK_DIR / "token_list.txt"
TOKEN_LIST.write_text("\n".join(s2t.converter.token_list))

tokenizer = s2t.tokenizer
converter = s2t.converter

def tokenize(text):
    return np.array(converter.tokens2ids(tokenizer.text2tokens(text)))

def detokenize(ids):
    return tokenizer.tokens2text(converter.ids2tokens(ids))



### Custom model wrapper

We need to define a class that will pass our pre-trained model to ESPnet

Related docs:
- [**Model component**](https://masao-someki.github.io/espnet_draft_home_page/espnet3/core/components/model.html)


In [ ]:
class OWSMBaseFinetuneModel(nn.Module):
    def __init__(
        self,
        *,
        model_tag: str,
        lang_sym: str,
        device: str = "cpu",
    ) -> None:
        super().__init__()
        s2t = Speech2Text.from_pretrained(
            model_tag=model_tag,
            lang_sym=lang_sym,
            device=str(device),
        )
        self.s2t_model = s2t.s2t_model

    def forward(self, **batch):
        return self.s2t_model(**batch)

    def collect_feats(self, **batch):
        return self.s2t_model.collect_feats(**batch)


## 5) Dataset

### Custom dataset

We need to convert the HuggingFace data into a format that ESPnet can read.
That is a PyTorch `Dataset` whose items carry the fields OWSM's preprocessor
expects.

**It goes in a file, not in this notebook.** ESPnet3 does not take a dataset
object: `DataOrganizer` resolves each entry in the config through
`load_dataset_module()`, which imports `<recipe_dir>/dataset/__init__.py` and
looks for a class called `Dataset`. The config then names that class's
arguments under `data_src_args`. A recipe in `egs3/` is laid out the same way,
so what you write here is what you would write there.

Related docs:
- [**Create dataset stage**](https://masao-someki.github.io/espnet_draft_home_page/espnet3/stages/create-dataset.html)
- [**Trainer config**](https://masao-someki.github.io/espnet_draft_home_page/espnet3/config/train_config.html)
- [**Data organizer**](https://masao-someki.github.io/espnet_draft_home_page/espnet3/core/components/data-organizer.html)

In [ ]:
from pathlib import Path

Path("dataset").mkdir(exist_ok=True)

In [ ]:
%%writefile dataset/__init__.py
"""One language of FLEURS, in the fields ESPnet's S2T preprocessor expects.

ESPnet3 imports this file: the class has to be called `Dataset`, and its
arguments are what the training config passes as `data_src_args`.
"""

import io

import numpy as np
import soundfile
from datasets import Audio as AudioFeature
from datasets import load_dataset
from torch.utils.data import Dataset as TorchDataset


class Dataset(TorchDataset):
    def __init__(self, *, data_lang: str, split: str, lang_sym: str,
                 task_sym: str) -> None:
        self.lang_sym = str(lang_sym)
        self.task_sym = str(task_sym)
        # decode=False keeps the audio as bytes: decoding through datasets
        # needs torchcodec, and soundfile reads the bytes directly
        self._ds = load_dataset(
            "google/fleurs", data_lang, split=str(split)
        ).cast_column("audio", AudioFeature(decode=False))

    def __len__(self) -> int:
        return len(self._ds)

    def __getitem__(self, idx: int):
        ex = self._ds[int(idx)]
        speech, _ = soundfile.read(io.BytesIO(ex["audio"]["bytes"]), dtype="float32")
        transcription = str(ex["transcription"]).strip()
        text = f"{self.lang_sym}{self.task_sym}<notimestamps> {transcription}"
        return {
            "speech": speech.astype(np.float32),
            "text": text,
            "text_prev": "<na>",
            "text_ctc": transcription,
        }

## 6) Training config in YAML

### Config

Training requires tuning many hyper-parameters. We have provided an initial config here to start you off.

The model class is ours and lives in this notebook, so the config names it
as `__main__.OWSMBaseFinetuneModel`. The dataset is named differently: the
organizer imports `dataset/__init__.py` from `recipe_dir` and passes each
entry's `data_src_args` to its `Dataset` class.

<!-- Edit `max_epochs`, `batch_bins`, or `limit_train_batches` for quick tests.-->

Related docs:
- [**Systems overview**](https://masao-someki.github.io/espnet_draft_home_page/espnet3/core/systems.html)
- [**Train stage**](https://masao-someki.github.io/espnet_draft_home_page/espnet3/stages/train.html)
- [**Training config**](https://masao-someki.github.io/espnet_draft_home_page/espnet3/config/train_config.html)


🔭 Exploration point:
*   Trying different configurations, such as lr, epochs


In [ ]:
EXP_TAG = f"owsm_v31_base_fleurs_{FLEURS_CONFIG}"
EXP_DIR = (WORK_DIR / "exp" / EXP_TAG).as_posix()
STATS_DIR = (WORK_DIR / "exp" / "stats").as_posix()

LANG_SYM = f"<{OWSM_LANG}>"
TASK_SYM = "<asr>"

yaml_cfg = f"""
num_device: 1
num_nodes: 1
exp_tag: {EXP_TAG}
recipe_dir: .
data_dir: {WORK_DIR / 'data'}
exp_dir: {EXP_DIR}
stats_dir: {STATS_DIR}
dataset_dir: {WORK_DIR / 'hf_cache'}

dataset:
  _target_: espnet3.components.data.data_organizer.DataOrganizer
  # where the organizer looks for dataset/__init__.py, whose Dataset class it
  # builds with each entry's data_src_args
  recipe_dir: .
  train:
    - name: train
      data_src_args:
        split: {TRAIN_SPLIT}
        data_lang: {FLEURS_CONFIG}
        lang_sym: {LANG_SYM}
        task_sym: {TASK_SYM}
  valid:
    - name: validation
      data_src_args:
        split: {VALID_SPLIT}
        data_lang: {FLEURS_CONFIG}
        lang_sym: {LANG_SYM}
        task_sym: {TASK_SYM}
  preprocessor:
    _target_: espnet2.train.preprocessor.S2TPreprocessor
    train: true
    token_type: bpe
    token_list: {TOKEN_LIST}
    bpemodel: {BPEMODEL}
    text_prev_name: text_prev
    text_ctc_name: text_ctc
    fs: 16000

dataloader:
  collate_fn:
    _target_: espnet2.train.collate_fn.CommonCollateFn
    int_pad_value: -1
  train:
    iter_factory:
      _target_: espnet2.iterators.sequence_iter_factory.SequenceIterFactory
      shuffle: true
      collate_fn: ${{dataloader.collate_fn}}
      num_workers: 0
      batches:
        type: numel
        shape_files:
          - ${{stats_dir}}/train/feats_shape
        batch_size: 2
        batch_bins: 1000000
  valid:
    iter_factory:
      _target_: espnet2.iterators.sequence_iter_factory.SequenceIterFactory
      shuffle: false
      collate_fn: ${{dataloader.collate_fn}}
      num_workers: 0
      batches:
        type: numel
        shape_files:
          - ${{stats_dir}}/valid/feats_shape
        batch_size: 2
        batch_bins: 2000000

model:
  _target_: __main__.OWSMBaseFinetuneModel
  model_tag: {MODEL_TAG}
  lang_sym: {LANG_SYM}
  device: {DEVICE}

optimizer:
  _target_: torch.optim.Adam
  lr: 3.0e-5

scheduler:
  _target_: torch.optim.lr_scheduler.StepLR
  step_size: 1000

best_model_criterion:
  - - valid/acc
    - 3
    - max

trainer:
  accelerator: auto
  devices: 1
  max_epochs: 1
  log_every_n_steps: 1
  limit_train_batches: {LIMIT_TRAIN_BATCHES}

fit: {{}}
"""

cfg = OmegaConf.create(yaml_cfg)
print("Config ready. exp_dir:", cfg.exp_dir)


## 7) Train (5-6 minutes)

### collect_stats + train

Finally, we pass the config to the trainer and start training.
ESPnet3’s training framework is built on top of the PyTorch Lightning Trainer.

Related docs:
- [**Collect Stats stage**](https://masao-someki.github.io/espnet_draft_home_page/espnet3/stages/collect-stats.html)
- [**Train stage**](https://masao-someki.github.io/espnet_draft_home_page/espnet3/stages/train.html)
- [**Training config**](https://masao-someki.github.io/espnet_draft_home_page/espnet3/config/train_config.html)


In [ ]:
from espnet3.systems.base.system import BaseSystem
from espnet3.utils.logging_utils import configure_logging
from espnet3.utils.stages_utils import run_stages

log = configure_logging()
system = BaseSystem(training_config=cfg)

run_stages(system, ["collect_stats", "train"], log=log)


## 8) Inference

Here is a demo of how to perform inference, and how to load checkpoints.


### Inference with original pre-trained model


In [ ]:
# the same class the trainer used, imported from the module it lives in
from dataset import Dataset as FLEURSDataset

test_dataset = FLEURSDataset(
    data_lang=FLEURS_CONFIG, split=TEST_SPLIT, lang_sym=LANG_SYM, task_sym=TASK_SYM
)
sample_test_utterance = test_dataset[0]

In [ ]:
import torch
_device="cuda" if torch.cuda.is_available() else "cpu" if torch.cuda.is_available() else "cpu"
s2t.s2t_model.to(_device)
s2t.device = _device

d = torch.load("original.pth")
s2t.s2t_model.load_state_dict(d)
pred = s2t(sample_test_utterance['speech'])
print('PREDICTED: ' + pred[0][0])
print('REFERENCE: ' + sample_test_utterance['text_ctc'])

### Inference with fine-tuned model


In [ ]:
# the last checkpoint the trainer wrote, rather than a step number spelled
# out here: the number follows limit_train_batches and max_epochs
checkpoint = max(
    Path(EXP_DIR).glob("step*.ckpt"), key=lambda p: int(p.stem.removeprefix("step"))
)
print("loading", checkpoint.name)
d = torch.load(checkpoint, map_location="cpu")
s2t.s2t_model.load_state_dict({
    k.replace("s2t_model.", "", 1): v
    for k, v in d["state_dict"].items()
})
pred = s2t(sample_test_utterance['speech'])
print('PREDICTED: ' + pred[0][0])
print('REFERENCE: ' + sample_test_utterance['text_ctc'])